In [ ]:
import os 
import tensorflow as tf
import time
import numpy as np
from glob import glob
from tqdm import tqdm
import cv2
from tensorflow import keras
from tensorflow.keras import layers
from imgaug import augmenters as iaa

In [ ]:
os.environ["PYTHONHASHSEED"] = str(42)
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
dataset_path = "resources"
save_path = "resources/predictions/non-aug"
save_path_aug = "resources/predictions/aug"


def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

create_dir(save_path_aug)
create_dir(save_path)


In [ ]:
def get_model(img_size, num_classes):
    inputs = keras.Input(shape=img_size + (3,))

    x = layers.Conv2D(32, 3, strides=2, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    previous_block_activation = x 
    for filters in [64, 128, 256]:
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

        residual = layers.Conv2D(filters, 1, strides=2, padding="same")(
            previous_block_activation
        )
        x = layers.add([x, residual])  
        previous_block_activation = x  
    for filters in [256, 128, 64, 32]:
        x = layers.Activation("relu")(x)
        x = layers.Conv2DTranspose(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.Activation("relu")(x)
        x = layers.Conv2DTranspose(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.UpSampling2D(2)(x)

        residual = layers.UpSampling2D(2)(previous_block_activation)
        residual = layers.Conv2D(filters, 1, padding="same")(residual)
        x = layers.add([x, residual])  
        previous_block_activation = x  

    outputs = layers.Conv2D(num_classes, 3, activation="softmax", padding="same")(x)

    model = keras.Model(inputs, outputs)
    return model

In [ ]:
model = get_model((768, 512), 2)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:
model.summary()

In [ ]:
test_X = glob(os.path.join(dataset_path, "test", "images", "*.png"))
test_Y = glob(os.path.join(dataset_path, "test", "masks", "*.png"))

train_X = glob(os.path.join(dataset_path, "train", "images", "*.png"))
train_Y = glob(os.path.join(dataset_path, "train", "masks", "*.png"))

In [ ]:
img_size = (768, 512)
num_classes = 2  
model = get_model(img_size, num_classes)

time_taken = []

for x in tqdm(train_X):
    name = x.split("/")[-1]
    x = cv2.imread(x, cv2.IMREAD_GRAYSCALE)
    x = cv2.resize(x, (img_size[1], img_size[0]))  
    x = np.stack([x] * 3, axis=-1)  
    x = x / 255.0
    start_time = time.time()
    p = model.predict(np.expand_dims(x, axis=0))
    total_time = time.time() - start_time
    time_taken.append(total_time)

    p = np.argmax(p, axis=-1)
    p = np.squeeze(p)


    p = p*255
    cv2.imwrite(os.path.join(save_path, name), p)
    

In [ ]:
train_aug_img_path = os.path.join(dataset_path, "train_aug", "images")
train_aug_mask_path = os.path.join(dataset_path, "train_aug", "masks")
test_aug_img_path = os.path.join(dataset_path, "test_aug", "images")
test_aug_mask_path = os.path.join(dataset_path, "test_aug", "masks")

In [ ]:
seq = iaa.Sequential([
    iaa.Fliplr(0.5),  
    iaa.Crop(percent=(0, 0.1)),  
    iaa.Sometimes(0.5,
        iaa.GaussianBlur(sigma=(0, 0.5))
    ),
    iaa.LinearContrast((0.75, 1.5)),
    iaa.AdditiveGaussianNoise(scale=(0, 0.05*255)),
    iaa.Multiply((0.8, 1.2)),
    iaa.Affine(
        translate_percent={"x": (-0.1, 0.1), "y": (-0.1, 0.1)},
        rotate=(-25, 25),
        shear=(-8, 8)
    )
])

num_augmented_images = 35

In [ ]:
def augment_and_save(images, masks, save_img_path, save_mask_path, num_aug=25):
    for img_path, mask_path in tqdm(zip(images, masks), total=len(images)):
        img = cv2.imread(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        for i in range(num_aug):
            img_aug, mask_aug = seq(image=img, segmentation_maps=mask[np.newaxis, :, :, np.newaxis])
            mask_aug = mask_aug[0, :, :, 0] 

            img_name = os.path.splitext(os.path.basename(img_path))[0] + f"_aug_{i}.png"
            mask_name = os.path.splitext(os.path.basename(mask_path))[0] + f"_aug_{i}.png"
            
            cv2.imwrite(os.path.join(save_img_path, img_name), img_aug)
            cv2.imwrite(os.path.join(save_mask_path, mask_name), mask_aug)

In [ ]:
augment_and_save(train_X, train_Y, train_aug_img_path, train_aug_mask_path)
augment_and_save(test_X, test_Y, test_aug_img_path, test_aug_mask_path)

In [ ]:
test_X = glob(os.path.join(dataset_path, "test_aug", "images", "*.png"))
test_Y = glob(os.path.join(dataset_path, "test_aug", "masks", "*.png"))

train_X = glob(os.path.join(dataset_path, "train_aug", "images", "*.png"))
train_Y = glob(os.path.join(dataset_path, "train_aug", "masks", "*.png"))

In [ ]:
img_size = (768, 512)
num_classes = 2  
model = get_model(img_size, num_classes)

time_taken = []

for x in tqdm(train_X):
    name = x.split("/")[-1]
    x = cv2.imread(x, cv2.IMREAD_GRAYSCALE)
    x = cv2.resize(x, (img_size[1], img_size[0]))  
    x = np.stack([x] * 3, axis=-1)  
    x = x / 255.0
    start_time = time.time()
    p = model.predict(np.expand_dims(x, axis=0))
    total_time = time.time() - start_time
    time_taken.append(total_time)

    p = np.argmax(p, axis=-1)
    p = np.squeeze(p)


    p = p*255
    cv2.imwrite(os.path.join(save_path_aug, name), p)
    